# LightGBM으로 작물 예측

토양의 질소(N)·인(P)·칼륨(K) 함량 비율과 pH, 기온(°C), 상대습도(%), 강우량(mm)을 입력해 작물 종류를 분류합니다. [Kaggle Crop Recommendation Dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset/data)의 `label`이 실제 작물 이름입니다.

Colab에서 **런타임 → 모두 실행**을 선택하세요. 다른 조건으로 예측하려면 마지막 셀의 입력값을 바꿔 다시 실행하세요.

In [1]:
%pip -q install kagglehub lightgbm pandas scikit-learn

## 1. 데이터 불러오기

`kagglehub`로 최신 버전을 내려받습니다. 이미 받은 `Crop_recommendation.csv`가 있다면 Colab의 파일 패널에 업로드해 사용할 수 있습니다.

In [2]:
from pathlib import Path

import kagglehub
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

dataset = 'atharvaingle/crop-recommendation-dataset'
csv_path = Path('Crop_recommendation.csv')
if not csv_path.is_file():
    csv_path = Path(kagglehub.dataset_download(dataset)) / csv_path.name

df = pd.read_csv(csv_path)
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[features]
y = df['label']

print(f'데이터 {len(df):,}개 · 작물 {y.nunique()}종')
display(df.head())

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
데이터 2,200개 · 작물 22종


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


## 2. 데이터 분할 및 모델 학습

작물별 비율을 유지하도록 데이터를 학습용 80%와 테스트용 20%로 나눕니다. LightGBM 모델은 학습용 데이터로 훈련합니다.

In [3]:
random_state = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=random_state
)

model = LGBMClassifier(
    objective='multiclass', random_state=random_state, n_jobs=2, verbosity=-1
)
model.fit(X_train, y_train)
print(f'학습용 {len(X_train):,}개 · 테스트용 {len(X_test):,}개')

학습용 1,760개 · 테스트용 440개


## 3. 테스트 세트 평가

- **정확도(Accuracy)**는 테스트 샘플 중 작물을 맞힌 비율입니다.
- **Macro F1**은 작물마다 F1 점수를 구한 뒤 같은 가중치로 평균한 값입니다.

In [4]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f'정확도 (Accuracy): {accuracy:.2%}')
print(f'Macro F1: {macro_f1:.4f}')

정확도 (Accuracy): 98.86%
Macro F1: 0.9886


## 4. 새 토양·기후 조건으로 예측하기

아래 예시의 일곱 값을 원하는 조건으로 바꾼 뒤 이 셀을 실행하세요.

In [ ]:
conditions = {
    'N': 90,
    'P': 42,
    'K': 43,
    'temperature': 25.0,  # °C
    'humidity': 82.0,     # %
    'ph': 6.5,
    'rainfall': 220.0,   # mm
}
sample = pd.DataFrame([conditions], columns=features)
predicted_crop = model.predict(sample)[0]
print(f'예측 작물: {predicted_crop}')